# Adversarial Examples

> Based on Stanford CS230 — LN4, Goodfellow et al. (2014)

A neural network that achieves 99 % accuracy can be fooled into misclassifying an image by adding an imperceptibly small, carefully crafted perturbation. These **adversarial examples** expose a fundamental brittleness in deep models.

## Learning Objectives

1. Define adversarial examples and explain why they exist
2. Implement the Fast Gradient Sign Method (FGSM)
3. Understand targeted vs untargeted attacks
4. Survey defences: adversarial training, input transformations, certified defences

## Why Adversarial Examples Exist

Neural networks partition input space with high-dimensional, nearly linear decision boundaries. A perturbation in the direction of the **gradient of the loss w.r.t. the input** moves the example across a boundary — even if the pixel change is invisible to humans.

## Fast Gradient Sign Method (FGSM)

$$x_{\text{adv}} = x + \epsilon \cdot \text{sign}\!\bigl(\nabla_x \mathcal{L}(\theta, x, y)\bigr)$$

- $\epsilon$ — perturbation magnitude (e.g. 0.01–0.1 in $[0,1]$ pixel scale)
- $\nabla_x \mathcal{L}$ — gradient of cross-entropy loss w.r.t. input pixels
- One gradient step → fast; iterative extensions (PGD) repeat for stronger attacks

## Attack Types

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 160" width="640" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="aa" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Untargeted -->
  <rect x="10" y="20" width="140" height="50" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="80" y="43" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Clean image x</text>
  <text x="80" y="60" text-anchor="middle" fill="#3a7bbf" font-size="9">True class: "panda"</text>
  <line x1="150" y1="45" x2="210" y2="45" stroke="#aaa" stroke-width="1.4" marker-end="url(#aa)"/>
  <text x="180" y="40" text-anchor="middle" fill="#888" font-size="9">+ε·sign(∇L)</text>
  <rect x="215" y="20" width="140" height="50" rx="5" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.5"/>
  <text x="285" y="43" text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">Adversarial x_adv</text>
  <text x="285" y="60" text-anchor="middle" fill="#b03a3a" font-size="9">Predicted: "gibbon" ✗</text>
  <text x="185" y="95" text-anchor="middle" fill="#888" font-size="10" font-style="italic">Untargeted attack — any wrong class</text>

  <!-- Targeted -->
  <rect x="370" y="20" width="140" height="50" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="440" y="43" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Clean image x</text>
  <text x="440" y="60" text-anchor="middle" fill="#3a7bbf" font-size="9">True class: "panda"</text>
  <line x1="510" y1="45" x2="570" y2="45" stroke="#aaa" stroke-width="1.4" marker-end="url(#aa)"/>
  <text x="540" y="40" text-anchor="middle" fill="#888" font-size="9">-ε·sign(∇L_target)</text>
  <rect x="575" y="20" width="55" height="50" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.5"/>
  <text x="602" y="43" text-anchor="middle" fill="#8a3ab8" font-size="10" font-weight="bold">x_adv</text>
  <text x="602" y="58" text-anchor="middle" fill="#8a3ab8" font-size="9">"cat" ✗</text>
  <text x="480" y="95" text-anchor="middle" fill="#888" font-size="10" font-style="italic">Targeted attack — specific wrong class</text>

  <!-- Defence note -->
  <rect x="10" y="118" width="620" height="30" rx="5" fill="#7ecba1" fill-opacity="0.12" stroke="#7ecba1" stroke-width="1.3"/>
  <text x="320" y="137" text-anchor="middle" fill="#3d7a5e" font-size="10">Defence: Adversarial Training — augment training set with adversarial examples and re-train</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Toy 2-class network to demonstrate FGSM ----
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def cross_entropy(probs, y_true):
    return -np.log(probs[y_true] + 1e-10)

def forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = np.tanh(z1)
    z2 = W2 @ a1 + b2
    return softmax(z2), a1, z1

def input_gradient(x, y_true, W1, b1, W2, b2):
    """Compute ∂L/∂x via backprop."""
    probs, a1, z1 = forward(x, W1, b1, W2, b2)
    dz2 = probs.copy(); dz2[y_true] -= 1          # (n_y,)
    da1 = W2.T @ dz2                               # (n_h,)
    dz1 = da1 * (1 - a1**2)                        # tanh derivative
    dx  = W1.T @ dz1                               # (n_x,)
    return dx, cross_entropy(probs, y_true)

def fgsm(x, y_true, W1, b1, W2, b2, eps):
    grad, loss = input_gradient(x, y_true, W1, b1, W2, b2)
    return x + eps * np.sign(grad), loss

# Initialise a small network
np.random.seed(3)
n_x, n_h, n_y = 8, 16, 4
W1 = np.random.randn(n_h, n_x) * 0.3
b1 = np.zeros(n_h)
W2 = np.random.randn(n_y, n_h) * 0.3
b2 = np.zeros(n_y)
x_clean = np.random.randn(n_x)
y_true  = 1

probs_clean, _, _ = forward(x_clean, W1, b1, W2, b2)
print(f"Clean prediction:        class {probs_clean.argmax()}  (true={y_true})")

# FGSM at multiple epsilon values
epsilons = [0.0, 0.05, 0.1, 0.2, 0.4, 0.8, 1.0]
results = []
for eps in epsilons:
    x_adv = fgsm(x_clean, y_true, W1, b1, W2, b2, eps)[0] if eps > 0 else x_clean
    probs_adv, _, _ = forward(x_adv, W1, b1, W2, b2)
    correct = probs_adv.argmax() == y_true
    results.append((eps, probs_adv[y_true], correct))
    print(f"ε={eps:.2f}  P(true class)={probs_adv[y_true]:.4f}  correct={correct}")

# ---- Iterative PGD attack (multi-step FGSM with clipping) ----
def pgd_attack(x, y_true, W1, b1, W2, b2, eps, alpha, n_steps):
    x_adv = x.copy()
    for _ in range(n_steps):
        grad, _ = input_gradient(x_adv, y_true, W1, b1, W2, b2)
        x_adv   = x_adv + alpha * np.sign(grad)
        x_adv   = np.clip(x_adv, x - eps, x + eps)   # project back to ε-ball
    return x_adv

x_pgd  = pgd_attack(x_clean, y_true, W1, b1, W2, b2, eps=0.3, alpha=0.05, n_steps=20)
probs_pgd, _, _ = forward(x_pgd, W1, b1, W2, b2)
print(f"\nPGD (20 steps) P(true class) = {probs_pgd[y_true]:.4f}  correct={probs_pgd.argmax()==y_true}")

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('FGSM Adversarial Attack on a Toy Network', fontsize=13, fontweight='bold')

# P(true class) vs epsilon
eps_vals = [r[0] for r in results]
p_true   = [r[1] for r in results]
axes[0].plot(eps_vals, p_true, 'o-', color=P[0], lw=2, ms=7)
axes[0].axhline(0.25, color='#bbb', ls='--', lw=1, label='Chance (1/4)')
axes[0].set_xlabel('Perturbation ε')
axes[0].set_ylabel('P(true class)')
axes[0].set_title('Confidence Degradation (FGSM)')
axes[0].legend(); axes[0].grid(True)

# Perturbation visualisation (2-D slice)
xx, yy = np.meshgrid(np.linspace(-3, 3, 120), np.linspace(-3, 3, 120))
Xg = np.c_[xx.ravel(), yy.ravel()]
np.random.seed(5)
W1s = np.random.randn(n_h, 2) * 0.3
probs_grid = np.array([forward(Xg[i], W1s, b1, W2, b2)[0].argmax() for i in range(len(Xg))])
axes[1].contourf(xx, yy, probs_grid.reshape(xx.shape), alpha=0.3, cmap='Set3')
x2 = np.array([0.5, -0.3])
grad2, _ = input_gradient(np.pad(x2, (0, n_x-2)), y_true, W1, b1, W2, b2)
axes[1].scatter(*x2, color=P[3], s=120, zorder=5, label='Clean x')
axes[1].annotate('', xy=x2 + 0.5*np.sign(grad2[:2]),
                 xytext=x2,
                 arrowprops=dict(arrowstyle='->', color=P[1], lw=2.5))
axes[1].set_title('FGSM direction in input space (2D slice)')
axes[1].set_xlabel('x₁'); axes[1].set_ylabel('x₂')
axes[1].legend(); axes[1].grid(True)

# FGSM vs PGD — perturbation norm
steps = np.arange(1, 21)
norms_pgd = []
x_adv_track = x_clean.copy()
for _ in steps:
    grad, _ = input_gradient(x_adv_track, y_true, W1, b1, W2, b2)
    x_adv_track = np.clip(x_adv_track + 0.05 * np.sign(grad), x_clean-0.3, x_clean+0.3)
    p, _, _ = forward(x_adv_track, W1, b1, W2, b2)
    norms_pgd.append(p[y_true])
axes[2].plot(steps, norms_pgd, 'o-', color=P[1], lw=2, ms=6, label='PGD (20 steps)')
axes[2].axhline(probs_clean[y_true], color=P[3], ls='--', lw=1.5, label='Clean confidence')
axes[2].axhline(0.25, color='#bbb', ls=':', lw=1, label='Chance')
axes[2].set_xlabel('PGD iteration')
axes[2].set_ylabel('P(true class)')
axes[2].set_title('PGD Attack Progress')
axes[2].legend(fontsize=9); axes[2].grid(True)

plt.tight_layout()
plt.show()
